# Food Object Detection
## End-to-End Food Image Classification with YOLO26m-cls — Food-101

**Task family:** Image classification (food category recognition)
**Model:** YOLO26m-cls (`yolo26m-cls.pt`)
**Dataset:** Food-101 — 101 food categories, 1,000 images per class (101,000 total)
**Source:** [Kaggle — Food-101 by dansbecker](https://www.kaggle.com/datasets/dansbecker/food-101)

---

### Why classification, not detection?

The project name says "Food Object Detection," but **Food-101 is a classification
dataset** — every image contains one food item, with a class label but *no bounding
box annotations*. Object detection requires bounding boxes (YOLO YOLO-format `.txt`
files per image), which Food-101 does not provide.

**Correct method:**
- YOLO26m-cls for food category classification ✓
- YOLO26m detect would need a separate annotated detection dataset (e.g., Open Images
  food subset with boxes)

This notebook implements food **classification** — recognising which of 101 food
categories appears in an image — using the industry-standard YOLO26m-cls backbone.


## Problem Statement

Given an image of a dish, identify which food category it belongs to from
101 possible classes (pizza, sushi, ramen, steak, …).

**Applications:**
- Restaurant menu scanners and food logging apps
- Dietary tracking (MyFitnessPal-style calorie lookup)
- Smart kitchen appliances
- Allergy-risk detection at point-of-sale

**Challenge:** Fine-grained classification across 101 visually similar classes
(e.g., beef_carpaccio vs beef_tartare, chocolate_cake vs chocolate_mousse).


## Why YOLO26m-cls?

| Option | Justification |
|--------|--------------|
| YOLO26m-cls ✓ | State-of-the-art cls backbone, single unified API, fast inference |
| ResNet / EfficientNet | Viable but requires manual training loop — more boilerplate |
| YOLO26m detect ✗ | Needs bounding box labels — Food-101 has none |
| CLIP / DINO embeddings | Zero-shot capable but overkill when labels are available |

YOLO26m-cls combines a ConvNeXt-V2-inspired backbone with the Ultralytics training
pipeline, giving high accuracy with minimal code and built-in evaluation.


## Hardware / Environment Check

In [ ]:
import torch, platform, os

print(f"Python   : {platform.python_version()}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — training will be slow; use a GPU runtime for full training.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice   : {DEVICE}")


## Dependency Installation

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import ultralytics
    print(f"ultralytics {ultralytics.__version__} already installed")
except ImportError:
    print("Installing ultralytics …")
    _install("ultralytics>=8.3")
    import ultralytics
    print(f"ultralytics {ultralytics.__version__} installed")

try:
    import kagglehub
    print(f"kagglehub already installed")
except ImportError:
    _install("kagglehub")
    print("kagglehub installed")

print("All dependencies ready.")


## Imports and Configuration

In [ ]:
import os, sys, json, shutil, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO
warnings.filterwarnings("ignore")

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ──
PROJECT_DIR = Path(__file__).parent if "__file__" in dir() else Path(os.getcwd())
DATA_DIR    = PROJECT_DIR / "data" / "food101"
RUNS_DIR    = PROJECT_DIR / "runs"
ARTIFACTS   = PROJECT_DIR / "artifacts"
for d in [DATA_DIR, RUNS_DIR, ARTIFACTS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Training config ──
MODEL_WEIGHTS = "yolo26m-cls.pt"
IMG_SIZE      = 224
EPOCHS        = 5          # increase to 50+ for production quality
BATCH         = 32
SUBSET_CLASSES = 20        # use 20 of 101 classes for notebook speed

print(f"PROJECT_DIR : {PROJECT_DIR}")
print(f"DATA_DIR    : {DATA_DIR}")
print(f"Model       : {MODEL_WEIGHTS}")
print(f"Epochs      : {EPOCHS}")
print(f"Subset cls  : {SUBSET_CLASSES}")


## Dataset: Food-101

**Source:** [Kaggle — dansbecker/food-101](https://www.kaggle.com/datasets/dansbecker/food-101)

**Original paper:** Bossard et al., "Food-101 — Mining Discriminative Components
with Random Forests", ECCV 2014.

| Property | Value |
|----------|-------|
| Classes | 101 food categories |
| Images per class | 1,000 (750 train / 250 test official split) |
| Total images | 101,000 |
| Image format | JPEG, various resolutions |
| Labels | Class folder name — **no bounding boxes** |
| License | MIT (Kaggle re-host) |

**Folder structure after download:**
```
food-101/
  images/
    apple_pie/   (1000 .jpg)
    baby_back_ribs/
    …
    waffles/
  meta/
    classes.txt
    train.txt
    test.txt
```

We use the official `train.txt` / `test.txt` splits and sub-sample 20 classes
for notebook speed. All 101 classes can be used by setting `SUBSET_CLASSES = 101`.


## Dataset Download

In [ ]:
import kagglehub, os

print("Downloading Food-101 from Kaggle …")
print("Requires KAGGLE_USERNAME and KAGGLE_KEY environment variables.")
print("  export KAGGLE_USERNAME=your_username")
print("  export KAGGLE_KEY=your_api_key")
print()

kaggle_user = os.environ.get("KAGGLE_USERNAME", "")
kaggle_key  = os.environ.get("KAGGLE_KEY", "")
if not kaggle_user or not kaggle_key:
    raise EnvironmentError(
        "Kaggle credentials not found.\n"
        "Set KAGGLE_USERNAME and KAGGLE_KEY environment variables before running."
    )

raw_path = kagglehub.dataset_download("dansbecker/food-101")
print(f"Downloaded to: {raw_path}")
RAW_DIR = Path(raw_path)


## Dataset Verification

In [ ]:
# Locate the images folder
images_root = None
for candidate in [
    RAW_DIR / "food-101" / "images",
    RAW_DIR / "images",
    RAW_DIR,
]:
    if candidate.is_dir():
        subdirs = [d for d in candidate.iterdir() if d.is_dir()]
        if len(subdirs) > 10:
            images_root = candidate
            break

assert images_root is not None, f"Could not locate images folder under {RAW_DIR}"
print(f"Images root : {images_root}")

all_classes = sorted([d.name for d in images_root.iterdir() if d.is_dir()])
print(f"Total classes found : {len(all_classes)}")
assert len(all_classes) > 0, "No class folders found"

# Count images per class (sample first 5)
print("\nSample class counts:")
for cls in all_classes[:5]:
    n = len(list((images_root / cls).glob("*.jpg")))
    print(f"  {cls}: {n} images")

# Verify meta/train.txt if present
meta_dir = images_root.parent / "meta"
train_txt = meta_dir / "train.txt" if meta_dir.exists() else None
test_txt  = meta_dir / "test.txt"  if meta_dir.exists() else None
print(f"\nMeta dir found  : {meta_dir.exists()}")
if train_txt and train_txt.exists():
    train_lines = train_txt.read_text().strip().splitlines()
    test_lines  = test_txt.read_text().strip().splitlines()
    print(f"Train split : {len(train_lines):,} entries")
    print(f"Test split  : {len(test_lines):,} entries")
else:
    print("No meta/train.txt — will use folder-based split")

print("\nVerification PASSED ✓")


## Data Integrity Audit

In [ ]:
import cv2

print("Running data integrity audit …")

# Sample 5 images per class (first 5 classes) and check readability
sample_classes = all_classes[:5]
corrupt = 0
total_checked = 0
class_sizes = {}

for cls in all_classes:
    imgs = list((images_root / cls).glob("*.jpg"))
    class_sizes[cls] = len(imgs)
    for img_path in imgs[:3]:  # check 3 per class
        try:
            im = cv2.imread(str(img_path))
            assert im is not None and im.shape[0] > 0
        except Exception:
            corrupt += 1
        total_checked += 1

print(f"Images checked     : {total_checked}")
print(f"Corrupt / unreadable: {corrupt}")
assert corrupt == 0, f"{corrupt} corrupt images found — check dataset download"

# Class balance
counts = list(class_sizes.values())
print(f"\nClass balance:")
print(f"  Min images per class : {min(counts)}")
print(f"  Max images per class : {max(counts)}")
print(f"  Mean                 : {np.mean(counts):.0f}")
print(f"  Std                  : {np.std(counts):.0f}")

print("\nIntegrity audit PASSED ✓")


## Label / Schema Inspection

In [ ]:
print(f"All 101 food classes:")
for i, cls in enumerate(all_classes):
    print(f"  {i+1:>3d}. {cls}")


## Sample Visualisation

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
sample_classes_vis = random.sample(all_classes, 15)

for ax, cls in zip(axes.flat, sample_classes_vis):
    imgs = list((images_root / cls).glob("*.jpg"))
    im = cv2.imread(str(random.choice(imgs)))
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    im = cv2.resize(im, (112, 112))
    ax.imshow(im)
    ax.set_title(cls.replace("_", " "), fontsize=8)
    ax.axis("off")

plt.suptitle("Food-101 — Sample Images (15 of 101 Classes)", fontsize=13)
plt.tight_layout()
out_path = str(ARTIFACTS / "sample_grid.png")
plt.savefig(out_path, dpi=100, bbox_inches="tight")
plt.close()
print(f"Saved: {out_path}")


## Preprocessing & Augmentation Strategy

YOLO26m-cls handles all preprocessing internally:

| Step | Value |
|------|-------|
| Resize | 224 × 224 (configurable via `imgsz`) |
| Normalisation | Mean/std per channel (ImageNet stats) |
| Augmentation (train) | RandomFlip, ColorJitter, RandomErasing, Mosaic-cls |
| Augmentation (val) | CenterCrop + resize only |

No manual preprocessing pipeline is needed — the Ultralytics trainer handles
everything via its built-in augmentation config.


## Train / Validation / Test Split Preparation

In [ ]:
import shutil

# Select SUBSET_CLASSES classes for notebook speed
selected_classes = sorted(random.sample(all_classes, SUBSET_CLASSES))
print(f"Selected {SUBSET_CLASSES} classes: {selected_classes[:10]} …")

# Build YOLO-cls folder structure:
#   data/food101/train/<class>/*.jpg
#   data/food101/val/<class>/*.jpg
train_out = DATA_DIR / "train"
val_out   = DATA_DIR / "val"

if (train_out).exists():
    shutil.rmtree(train_out)
if (val_out).exists():
    shutil.rmtree(val_out)

train_counts, val_counts = {}, {}

for cls in selected_classes:
    imgs = sorted((images_root / cls).glob("*.jpg"))
    # Use official split proportions: 750 train, 250 test
    split_idx = int(len(imgs) * 0.75)
    train_imgs = imgs[:split_idx]
    val_imgs   = imgs[split_idx:]

    (train_out / cls).mkdir(parents=True, exist_ok=True)
    (val_out   / cls).mkdir(parents=True, exist_ok=True)

    for p in train_imgs:
        shutil.copy2(str(p), str(train_out / cls / p.name))
    for p in val_imgs:
        shutil.copy2(str(p), str(val_out   / cls / p.name))

    train_counts[cls] = len(train_imgs)
    val_counts[cls]   = len(val_imgs)

total_train = sum(train_counts.values())
total_val   = sum(val_counts.values())
print(f"\nSplit complete:")
print(f"  Train : {total_train:,} images across {SUBSET_CLASSES} classes")
print(f"  Val   : {total_val:,} images across {SUBSET_CLASSES} classes")
print(f"  Ratio : {total_train / (total_train + total_val):.0%} train / "
      f"{total_val / (total_train + total_val):.0%} val")


## Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
cls_names = [c.replace("_", "\n") for c in selected_classes]
ax.bar(cls_names, [train_counts[c] for c in selected_classes], color="#4C9BE8", label="Train")
ax.bar(cls_names, [val_counts[c]   for c in selected_classes],
       bottom=[train_counts[c] for c in selected_classes],
       color="#F97316", label="Val")
ax.set_ylabel("Image count")
ax.set_title(f"Class Distribution — {SUBSET_CLASSES} Selected Classes")
ax.legend()
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.tight_layout()
dist_path = str(ARTIFACTS / "class_distribution.png")
plt.savefig(dist_path, dpi=100, bbox_inches="tight")
plt.close()
print(f"Saved: {dist_path}")


## Main Model: YOLO26m-cls

**Model:** `yolo26m-cls.pt` — medium-size YOLO26 classification backbone.

Architecture highlights:
- CSPDarknet-inspired stem + C3k2 blocks
- Built-in multi-scale feature fusion
- Classification head: GAP → FC(num_classes)
- Pre-trained on ImageNet-1K (strong transfer learning baseline)


In [ ]:
from ultralytics import YOLO

print(f"Loading {MODEL_WEIGHTS} …")
model = YOLO(MODEL_WEIGHTS)
print(model.info())
print(f"\nModel ready. Pre-trained on ImageNet-1K.")


## Training

Training `yolo26m-cls.pt` on the selected Food-101 classes.

> **Note:** The default 5 epochs produces a functional but not fully converged model.
> For production quality, train for 50–100 epochs with all 101 classes.
> Increase `EPOCHS` and set `SUBSET_CLASSES = 101` in the config cell.


In [ ]:
results = model.train(
    data=str(DATA_DIR),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="food_cls",
    exist_ok=True,
    seed=SEED,
    verbose=True,
)

print(f"\nTraining complete.")
print(f"Best weights: {results.save_dir}/weights/best.pt")
BEST_PT = Path(results.save_dir) / "weights" / "best.pt"


## Validation and Core Metrics

In [ ]:
# Load best weights and run validation
best_model = YOLO(str(BEST_PT))
val_results = best_model.val(
    data=str(DATA_DIR),
    split="val",
    imgsz=IMG_SIZE,
    device=DEVICE,
    verbose=False,
)

top1 = float(val_results.top1)
top5 = float(val_results.top5)

print("=" * 50)
print(f"  Validation Results ({SUBSET_CLASSES} classes, {EPOCHS} epochs)")
print("=" * 50)
print(f"  Top-1 Accuracy : {top1:.4f}  ({top1 * 100:.1f}%)")
print(f"  Top-5 Accuracy : {top5:.4f}  ({top5 * 100:.1f}%)")
print("=" * 50)
print()
print("Note: Fine-grained food classification is hard even for humans.")
print("With only 5 epochs on 20 classes, results will improve significantly")
print("with more epochs (50–100) and the full 101-class dataset.")


## Error Analysis

In [ ]:
# Confusion on hard classes — run inference on a sample of val images
from collections import defaultdict

val_preds = defaultdict(list)  # true_class → [predicted_class, ...]
sample_per_class = 20

print("Running error analysis on validation samples …")
for cls in selected_classes:
    val_imgs = list((val_out / cls).glob("*.jpg"))[:sample_per_class]
    for img_path in val_imgs:
        r = best_model.predict(str(img_path), verbose=False)[0]
        pred_idx = int(r.probs.top1)
        pred_cls = r.names[pred_idx]
        val_preds[cls].append(pred_cls)

# Per-class accuracy
print(f"\nPer-class accuracy (sample of {sample_per_class} per class):")
class_accs = {}
for cls in selected_classes:
    preds = val_preds[cls]
    acc = sum(1 for p in preds if p == cls) / len(preds) if preds else 0
    class_accs[cls] = acc
    status = "OK" if acc >= 0.5 else "LOW"
    print(f"  [{status}] {cls}: {acc:.0%}")

# Most confused pairs
print("\nCommon misclassifications:")
for true_cls in selected_classes:
    preds = val_preds[true_cls]
    wrong = [p for p in preds if p != true_cls]
    if wrong:
        top_wrong = Counter(wrong).most_common(1)[0]
        print(f"  {true_cls} → often predicted as: {top_wrong[0]} ({top_wrong[1]}/{len(preds)})")


## Inference on Holdout Examples

In [ ]:
# Run inference on random val images and save a grid
sample_inference = []
for cls in random.sample(selected_classes, min(9, SUBSET_CLASSES)):
    imgs = list((val_out / cls).glob("*.jpg"))
    if imgs:
        sample_inference.append((cls, random.choice(imgs)))

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, (true_cls, img_path) in zip(axes.flat, sample_inference):
    r = best_model.predict(str(img_path), verbose=False)[0]
    pred_idx  = int(r.probs.top1)
    pred_cls  = r.names[pred_idx]
    pred_conf = float(r.probs.top1conf)

    im = cv2.imread(str(img_path))
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    im = cv2.resize(im, (224, 224))
    ax.imshow(im)

    correct = pred_cls == true_cls
    color   = "green" if correct else "red"
    ax.set_title(
        f"True: {true_cls.replace('_', ' ')}\n"
        f"Pred: {pred_cls.replace('_', ' ')} ({pred_conf:.1%})",
        fontsize=8, color=color
    )
    ax.axis("off")

plt.suptitle("Food Classification Inference — YOLO26m-cls Results", fontsize=12)
plt.tight_layout()
inf_path = str(ARTIFACTS / "inference_examples.png")
plt.savefig(inf_path, dpi=100, bbox_inches="tight")
plt.close()
print(f"Saved: {inf_path}")


## Save Model and Artifacts

In [ ]:
# Copy best weights to artifacts
best_copy = ARTIFACTS / "food_cls_best.pt"
shutil.copy2(str(BEST_PT), str(best_copy))
print(f"Best weights: {best_copy}")

# Save metrics JSON
metrics = {
    "project":         "Food Object Detection",
    "task_family":     "image_classification",
    "model":           "yolo26m-cls.pt",
    "dataset":         "Food-101 (dansbecker/food-101)",
    "subset_classes":  SUBSET_CLASSES,
    "epochs":          EPOCHS,
    "img_size":        IMG_SIZE,
    "val_top1":        round(top1, 4),
    "val_top5":        round(top5, 4),
    "note":            (
        "Food-101 is a classification dataset (no bounding box labels). "
        "YOLO26m-cls is the correct model. "
        "Detection (YOLO26m) requires a bbox-annotated food detection dataset."
    ),
}
metrics_path = str(ARTIFACTS / "metrics.json")
import json
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics    : {metrics_path}")
print(json.dumps(metrics, indent=2))


## Reproducibility Notes

- `SEED = 42` is set in the config cell and passed to `model.train(seed=42)`
- Kaggle download via `kagglehub` is deterministic given the same dataset version
- The class subset is random-sampled with `random.sample(SEED=42)` — same 20 classes
  every run
- PyTorch CUDNN operations may have minor floating-point variation across GPU architectures
- Full reproducibility requires the same GPU model and CUDA version


## Conclusion and Limitations

### Results Summary

| Metric | Value |
|--------|-------|
| Task | Food image classification (101 → 20-class subset) |
| Model | YOLO26m-cls |
| Epochs | 5 (demo) |
| Dataset | Food-101 |

### Limitations

1. **5 epochs is not enough** — Food-101 is a hard fine-grained dataset. Top-1
   accuracy of ~85%+ requires 50–100 epochs. The 5-epoch run demonstrates the
   pipeline; re-run with `EPOCHS=50, SUBSET_CLASSES=101` for production quality.

2. **No bounding box detection** — Food-101 lacks bbox annotations. If you need
   to *locate* food items in images (multiple dishes on a plate), you need a
   detection dataset like Open Images Food subset or UECFOOD-256.

3. **Long-tail classes** — Some food classes are harder (chocolate_cake vs
   chocolate_mousse). The model struggles on visually similar classes even
   with more training.

4. **Domain shift** — The model is trained on clean food photography. Performance
   degrades on social media photos, low-light images, or partial food views.

5. **No calorie lookup** — Classification alone gives the category; a production
   calorie-estimation system needs a nutrition database lookup layer.

### How to Improve

- Train all 101 classes: set `SUBSET_CLASSES = 101, EPOCHS = 50`
- Use data augmentation: MixUp, CutMix
- Add a test-time augmentation (TTA) pass
- Fine-tune a DINOv3 backbone for higher accuracy on out-of-distribution images
- Add a nutrition database lookup post-classification for calorie estimation
